In [1]:
# %pip install delta-spark

In [2]:
from pyspark.sql import SparkSession
from delta.tables import DeltaTable
from pyspark.sql.functions import col

In [3]:
existing_spark = SparkSession.getActiveSession()
print(existing_spark)
if existing_spark:
    print("Existing Spark session found. Stopping it...")
    existing_spark.stop()

None


In [4]:
spark = (
        SparkSession.builder
        .appName("DeltaTable App")
        .master("local[*]")
        .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.1.0,"
                                        "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.5.0,"
                                        "org.apache.hudi:hudi-spark3.5-bundle_2.12:0.15.0,"
                                        "org.apache.hadoop:hadoop-aws:3.3.4,"
                                        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
                                        "software.amazon.awssdk:glue:2.25.26,"
                                        "software.amazon.awssdk:sts:2.25.26,"
                                        "software.amazon.awssdk:s3:2.25.26,"
                                        "software.amazon.awssdk:dynamodb:2.25.26,"
                                        "org.apache.iceberg:iceberg-aws-bundle:1.5.0,"
                                        "software.amazon.awssdk:url-connection-client:2.25.26")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension,"
                                        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions,"
                                        "org.apache.spark.sql.hudi.HoodieSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
        .config("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
        .config("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
        .config("spark.sql.catalog.glue_catalog.warehouse", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.warehouse.dir", "s3a://shivchoudhury-datasets/warehouse/")
        .config("spark.sql.catalog.glue_catalog.client.region", "ap-south-1")
        .config("spark.sql.defaultCatalog", "spark_catalog")
        .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
        .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
        .config("spark.hadoop.fs.s3a.endpoint", "s3.ap-south-1.amazonaws.com")
        .config("spark.hadoop.fs.s3a.path.style.access", "true")
        .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer")
        .config("hive.metastore.client.factory.class", "com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory")
        .enableHiveSupport()
        .getOrCreate()
    )

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.hudi#hudi-spark3.5-bundle_2.12 added as a dependency
software.amazon.awssdk#glue added as a dependency
software.amazon.awssdk#sts added as a dependency
software.amazon.awssdk#s3 added as a dependency
software.amazon.awssdk#url-connection-client added as a dependency
software.amazon.awssdk#dynamodb added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-90153ae3-a53a-43ae-a0c8-5604befe2511;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.a

In [5]:
spark

26/04/26 19:40:17 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


In [18]:
data = [
    (1, "Alice", 30, "2025-12-01"),
    (2, "Bob",   28, "2025-12-01"),
    (3, "Cleo",  35, "2025-12-02"),
]
cols = ["id", "name", "age", "dt"]

df = spark.createDataFrame(data, cols)

In [13]:
spark.sql("SHOW CATALOGS").show(truncate=False)

+-------------+
|catalog      |
+-------------+
|spark_catalog|
+-------------+



In [14]:
spark.sql("USE spark_catalog")

DataFrame[]

In [15]:
spark.sql("SELECT current_catalog()").show()

+-----------------+
|current_catalog()|
+-----------------+
|    spark_catalog|
+-----------------+



In [19]:
df = df.withColumn("id", col("id").cast("long"))

In [21]:
spark.sql("DROP TABLE my_glue_db.delta_exp_tbl")

DataFrame[]

In [22]:
df.write.format("delta").mode("overwrite").partitionBy("dt").saveAsTable("my_glue_db.delta_exp_tbl")

26/04/26 19:45:47 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`my_glue_db`.`delta_exp_tbl` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [11]:
spark.sql("USE DATABASE my_glue_db")

26/04/26 19:41:37 WARN ObjectStore: Failed to get database global_temp, returning NoSuchObjectException


DataFrame[]

In [68]:
spark.sql("SHOW TABLES").show()

+----------+-------------+-----------+
| namespace|    tableName|isTemporary|
+----------+-------------+-----------+
|my_glue_db|delta_exp_tbl|      false|
|my_glue_db|    delta_src|      false|
+----------+-------------+-----------+



In [23]:
source_df = spark.createDataFrame([
    (1, "Alice", 30),
    (2, "Bob", 28),
    (4, "Dave", 35)
], ["id","name","age"])

In [24]:
spark.sql("DROP TABLE my_glue_db.delta_src")

DataFrame[]

In [25]:
source_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("my_glue_db.delta_src")

26/04/26 19:46:17 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider delta. Persisting data source table `spark_catalog`.`my_glue_db`.`delta_src` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.


In [26]:
if not DeltaTable.isDeltaTable(spark, spark.sql("DESCRIBE DETAIL my_glue_db.delta_src").collect()[0]['location']):
    target_initial = spark.createDataFrame([
        (1, "Alice", 29),   # age changed
        (2, "Bob", 28),
        (3, "Charlie", 40)
    ], ["id","name","age"])
    target_initial.write.format("delta").mode("overwrite").saveAsTable("my_glue_db.delta_src_tbl")

In [27]:
spark.sql("DESCRIBE DETAIL my_glue_db.delta_src").select("name", "location").show(truncate=False)

+----------------------------------+--------------------------------------------------------------+
|name                              |location                                                      |
+----------------------------------+--------------------------------------------------------------+
|spark_catalog.my_glue_db.delta_src|s3a://shivchoudhury-datasets/warehouse/my_glue_db.db/delta_src|
+----------------------------------+--------------------------------------------------------------+



In [28]:
delta_table = DeltaTable.forPath(spark, spark.sql("DESCRIBE DETAIL my_glue_db.delta_src").collect()[0]['location'])

In [29]:
delta_table.toDF().show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  4| Dave| 35|
|  2|  Bob| 28|
+---+-----+---+



In [30]:
delta_table.alias("t").merge(source_df.alias("s"), "t.id = s.id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [31]:
spark.sql("select * from my_glue_db.delta_src").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 30|
|  2|  Bob| 28|
|  4| Dave| 35|
+---+-----+---+



In [32]:
new_source_df = spark.createDataFrame([
    (1, "Alice", 50),
    (2, "Bob", 15),
    (5, "Rey", 25),
    (7, "Rashi", 30)
], ["id","name","age"])

In [33]:
delta_table.alias("t").merge(new_source_df.alias("s"), "t.id = s.id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [34]:
spark.sql("select * from my_glue_db.delta_src").show()

+---+-----+---+
| id| name|age|
+---+-----+---+
|  1|Alice| 50|
|  2|  Bob| 15|
|  4| Dave| 35|
|  5|  Rey| 25|
|  7|Rashi| 30|
+---+-----+---+



In [35]:
delta_table.delete("true") # deletes all data files but keeps metadata

In [36]:
spark.sql("DROP TABLE IF EXISTS my_glue_db.delta_src")
spark.sql("DROP TABLE IF EXISTS my_glue_db.delta_exp_tbl")

DataFrame[]